# MLPRegressor — abandono em t+1 (Etapa 4)

Espelho do exemplo **"MLPRegressor - House Sales Prediction"** (`docs/tese_base/`), nas mesmas seções e na mesma
ordem. Protocolo: `docs/PROTOCOLO.md`, escrito antes da execução.

**Este notebook só exibe** (`CLAUDE.md`, regra 4). Todo número vem pronto de
`experimentos/regressao_mlp_2013_2018/saida/`, gravado por `04_regressao_mlp.py`. Nenhuma célula ajusta modelo, calcula
métrica ou correlação. **A régua (o abandono de t−1) aparece em todos os gráficos e tabelas.**

**Estado:** esqueleto. O experimento ainda não rodou; as células param com aviso se as saídas não existirem.

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo importa as bibliotecas de modelo (`MLPRegressor`, `StandardScaler`, `train_test_split`, `metrics`) e monta o Google Drive. Aqui só entram leitura e gráfico: o modelo está no script do experimento.
> - `seaborn` não está no `requirements.txt`; os gráficos usam `matplotlib` e `src/figuras.py`.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "figuras.py").exists())
import sys; sys.path.insert(0, str(RAIZ))
from src import figuras as F

F.aplicar_estilo()
F.conferir_ambiente(RAIZ / "requirements.txt", ("pandas", "numpy", "matplotlib", "scikit-learn"))

SAIDA = RAIZ / "experimentos" / "regressao_mlp_2013_2018" / "saida"
ARQ = {nome: SAIDA / f"04_{nome}.csv" for nome in
       ("correlacao_treino", "variaveis", "previsoes", "amostra_grafico", "curva_perda",
        "cv_temporal", "metricas", "testes")}
F.exigir(ARQ, como_gerar="python experimentos/regressao_mlp_2013_2018/04_regressao_mlp.py (execução única, "
                          "depois do protocolo no main)")
ETAPAS, CONFIGS = ("AF", "EM"), ("A", "B")

## 1. Carregamento e correlação

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo carrega o CSV inteiro e calcula `df.corr()` no conjunto todo, inclusive no que vira teste. Aqui a correlação é calculada **pelo script, só no treino** (t = 2013–2016), e o notebook só a desenha.
> - A matriz é **entre preditores**: a correlação com o desfecho não escolhe variável (regra 7). Se aparecer, é só exibição.
> - Um mapa por etapa e configuração, porque cada etapa só vê as próprias variáveis. A régua é `abandono_tm1` e vem marcada no eixo.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
corr = pd.read_csv(ARQ["correlacao_treino"])
fig, eixos = plt.subplots(2, 2, figsize=(11, 10))
for i, etapa in enumerate(ETAPAS):
    for j, cfg in enumerate(CONFIGS):
        m = (corr.query("etapa == @etapa and configuracao == @cfg")
                 .pivot(index="var_1", columns="var_2", values="r"))
        ax = eixos[i, j]
        im = ax.imshow(m.to_numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
        rot = [f"{c}  ← régua" if c == "abandono_tm1" else c for c in m.columns]
        ax.set_xticks(range(len(rot)), rot, rotation=90); ax.set_yticks(range(len(rot)), rot)
        F.titulo(ax, f"{etapa}, configuração {cfg}: correlação entre preditores (treino)", "MEDIDO")
fig.colorbar(im, ax=eixos, shrink=0.6, label="r de Pearson (treino)")
plt.show()

## 2. Exclusão de variáveis

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo exclui `id`, `date`, `yr_built`, `yr_renovated`, `zipcode`, `lat` e `long`, por pertinência. Aqui também: identificação, alvo, dados de t (não publicados quando a lista é feita) e redundâncias exatas (`docs/PROTOCOLO.md`, 4.1).
> - Acrescentamos a regra de redundância escrita antes de rodar: entre dois preditores com |r| > 0,90 no treino, sai o de menor prioridade (4.3). O desfecho nunca entra nessa conta.
> - Duas configurações: **A** (com abandono e reprovação de t−1) e **B** (sem nenhuma taxa de rendimento).
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
var = pd.read_csv(ARQ["variaveis"])
display(var.pivot_table(index="variavel", columns=["etapa", "configuracao"], values="situacao", aggfunc="first"))
display(var.loc[var["situacao"] == "sai", ["etapa", "configuracao", "variavel", "motivo"]])

In [ ]:
# Correlação após a exclusão (o segundo mapa do exemplo): só as variáveis que ficaram.
fica = var.loc[var["situacao"] == "fica"]
fig, eixos = plt.subplots(2, 2, figsize=(11, 10))
for i, etapa in enumerate(ETAPAS):
    for j, cfg in enumerate(CONFIGS):
        cols = set(fica.query("etapa == @etapa and configuracao == @cfg")["variavel"])
        m = (corr.query("etapa == @etapa and configuracao == @cfg and var_1 in @cols and var_2 in @cols")
                 .pivot(index="var_1", columns="var_2", values="r"))
        ax = eixos[i, j]
        im = ax.imshow(m.to_numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
        rot = [f"{c}  ← régua" if c == "abandono_tm1" else c for c in m.columns]
        ax.set_xticks(range(len(rot)), rot, rotation=90); ax.set_yticks(range(len(rot)), rot)
        F.titulo(ax, f"{etapa}, configuração {cfg}: variáveis que ficaram", "MEDIDO")
fig.colorbar(im, ax=eixos, shrink=0.6, label="r de Pearson (treino)")
plt.show()

## 3. Pré-processamento

> **O que mudamos em relação ao exemplo e por quê**
>
> - **Divisão:** o exemplo sorteia 30% das linhas para teste (`train_test_split`). Aqui a divisão é temporal e fixa: treino t = 2013–2016 (desfechos 2014–2017), teste t = 2017–2018 (desfechos 2018–2019). Os anos de teste já foram vistos na classificação de 2026-11-10, e nada é escolhido por eles.
> - **Padronização:** o exemplo ajusta o `StandardScaler` no conjunto todo, antes de dividir. Aqui ele fica dentro de um `Pipeline`, ajustado só no treino, depois da imputação pela mediana (a infraestrutura tem vazios) e com one-hot de localização e rede.
> - **Alvo:** o exemplo divide pelo máximo dos dados (`y.max()`), que inclui o teste. Aqui o alvo é dividido por 100, o domínio conhecido da taxa; as métricas voltam a pontos percentuais.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
# O pré-processamento roda no script; aqui só se mostra o tamanho de cada parte, lido das previsões.
prev = pd.read_csv(ARQ["previsoes"], dtype={"co_entidade": str})
display(prev.query("modelo == 'regua'")
            .groupby(["etapa", "configuracao", "ano_alvo"]).size().rename("escolas no teste").unstack())

## 4. Aplicação do MLPRegressor

> **O que mudamos em relação ao exemplo e por quê**
>
> - Os parâmetros são **os do exemplo**: `hidden_layer_sizes=50`, `activation='logistic'`, `solver='adam'`, `max_iter=5000`, `tol=1e-7`, `momentum=0.8` (sem efeito com `adam`), `early_stopping=True`, `n_iter_no_change=10`.
> - A topologia de 50 neurônios **segue o exemplo do professor**, não uma validação. 10 e 25 neurônios são só sensibilidade e nunca substituem o resultado principal.
> - O exemplo treina uma vez (`random_state=12`). Aqui são 10 sementes, relatando média e desvio.
> - A validação cruzada temporal (10, 25 e 50 neurônios, dentro do treino) é **só relatada**: não escolhe nada.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): 5.5.1 (validação cruzada) e 5.5.5 (várias execuções do treinamento).

In [ ]:
cv = pd.read_csv(ARQ["cv_temporal"])
display(cv.pivot_table(index=["etapa", "configuracao", "ano_validacao"], columns="topologia",
                       values=["mae_media", "mae_desvio"]))
# Mesma tabela da régua nas dobras de validação, para comparação.
display(cv.query("topologia == 'regua'")[["etapa", "configuracao", "ano_validacao", "mae_media"]])

## 5. Avaliação do modelo — desejado × estimado

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo mostra as 40 primeiras linhas do teste em barras. Aqui são 40 escolas-ano sorteadas pelo script com semente fixa (`04_amostra_grafico.csv`), e **a régua aparece ao lado** do desejado e do estimado.
> - O estimado do gráfico é o da semente 1, só para ilustrar; as métricas usam as 10 sementes.
> - Acrescentamos o gráfico de dispersão de todas as escolas de teste, porque 40 barras não mostram o padrão.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
amostra = pd.read_csv(ARQ["amostra_grafico"], dtype={"co_entidade": str})
for etapa in ETAPAS:
    for cfg in CONFIGS:
        a = amostra.query("etapa == @etapa and configuracao == @cfg")
        ax = a[["desejado", "rede_semente_1", "regua"]].plot(kind="bar", figsize=(10, 4), width=0.8)
        ax.set_xticks([]); ax.set_ylabel("abandono em t+1 (%)")
        ax.legend(["desejado", "estimado (rede, semente 1)", "régua (abandono de t−1)"])
        F.titulo(ax, f"{etapa}, configuração {cfg}: 40 escolas-ano de teste", "MEDIDO")
        plt.show()

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
for i, etapa in enumerate(ETAPAS):
    for j, cfg in enumerate(CONFIGS):
        p = prev.query("etapa == @etapa and configuracao == @cfg "
                       "and (modelo == 'regua' or (modelo == 'rede_50' and semente == 1))")
        ax = eixos[i, j]
        for modelo, cor, nome in (("regua", F.COR_SECUNDARIA, "régua"), ("rede_50", F.COR, "rede (50, semente 1)")):
            q = p.query("modelo == @modelo")
            ax.scatter(q["desejado"], q["estimado"], s=8, alpha=0.5, color=cor, label=nome)
        ax.axline((0, 0), slope=1, color="black", linewidth=0.8)
        ax.set_xlabel("desejado (%)"); ax.set_ylabel("estimado (%)"); ax.legend()
        F.titulo(ax, f"{etapa}, configuração {cfg}", "MEDIDO")
plt.show()

## 6. Avaliação do erro no processo de treinamento — curva de perda

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo mostra uma curva (`loss_curve_`). Aqui são as 10 sementes de cada etapa e configuração.
> - A linha horizontal é a perda da régua no treino, na mesma escala (metade do erro quadrático médio, alvo ÷ 100). A perda da rede inclui também o termo de regularização (`alpha`), então a comparação é aproximada.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): a conferir.

In [ ]:
perda = pd.read_csv(ARQ["curva_perda"])
fig, eixos = plt.subplots(2, 2, figsize=(11, 8))
for i, etapa in enumerate(ETAPAS):
    for j, cfg in enumerate(CONFIGS):
        p = perda.query("etapa == @etapa and configuracao == @cfg and topologia == 50")
        ax = eixos[i, j]
        for _, q in p.groupby("semente"):
            ax.plot(q["epoca"], q["perda"], color=F.COR, linewidth=0.8, alpha=0.6)
        ax.axhline(p["perda_regua_treino"].iloc[0], color=F.COR_SECUNDARIA, linestyle="--", label="régua (treino)")
        ax.set_xlabel("Épocas"); ax.set_ylabel("Custo"); ax.legend()
        F.titulo(ax, f"{etapa}, configuração {cfg}: curva de perda (10 sementes)", "MEDIDO")
plt.show()

## 7. Resumo das métricas

> **O que mudamos em relação ao exemplo e por quê**
>
> - O exemplo imprime MAE, MSE, RMSE (com `squared=False`), MAPE e R² de uma execução.
> - Aqui: MAE, RMSE (`root_mean_squared_error`, porque `squared=False` foi removido no scikit-learn 1.8) e R², com média e desvio nas 10 sementes e IC 95% por bootstrap de escolas; Recall@20% como complemento.
> - **Sem MAPE**: o abandono tem muitos zeros, e o erro percentual divide por eles.
> - Régua, B2 e Ridge na mesma tabela. A leitura ("supera a régua") segue a regra escrita antes: limite superior do IC da diferença de MAE abaixo de zero **e** p de Holm ≤ 0,05.
>
> **Livro** (Silva, Spatti e Flauzino, cap. 5): 5.5.5 (média e desvio das execuções).

In [ ]:
met = pd.read_csv(ARQ["metricas"])
ordem = ["regua", "B2", "ridge", "rede_50", "rede_25", "rede_10"]
for etapa in ETAPAS:
    for cfg in CONFIGS:
        t = (met.query("etapa == @etapa and configuracao == @cfg")
                .set_index("modelo").loc[lambda d: [m for m in ordem if m in d.index]])
        print(f"{etapa}, configuração {cfg}")
        display(t[["MAE_media", "MAE_desvio", "MAE_lo", "MAE_hi", "RMSE_media", "RMSE_lo", "RMSE_hi",
                   "R2_media", "R2_lo", "R2_hi", "Rec20_media", "Rec20_lo", "Rec20_hi"]])

In [ ]:
tes = pd.read_csv(ARQ["testes"])
display(tes.query("familia in ('F1', 'F2')")[["familia", "etapa", "configuracao", "comparacao",
                                               "dif", "lo", "hi", "p", "p_holm", "leitura"]])
fig, ax = plt.subplots(figsize=(7, 3.5))
f = tes.query("familia in ('F1', 'F2')")
F.intervalos(ax, (f["familia"] + " " + f["etapa"] + " " + f["configuracao"] + ": " + f["comparacao"]).tolist(),
             f["dif"], f["lo"], f["hi"])
ax.set_xlabel("diferença de MAE (pp); abaixo de zero = a rede erra menos")
F.titulo(ax, "Rede − régua (F1) e rede B − B2 (F2), IC 95%", "MEDIDO")
plt.show()

**Leitura.** Preenchida depois da execução, com os números de `04_testes.csv`, nunca digitados. Lembrete do
protocolo (seção 12): ganhar da régua no MAE não basta para dizer que a rede faz uma lista melhor; o Recall@20% ao
lado diz isso.